In [1]:
!pip install open3d

You should consider upgrading via the '/opt/conda/bin/python -m pip install --upgrade pip' command.


In [2]:
import open3d as o3d
import numpy as np
import os
from scipy.stats import special_ortho_group

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
def preprocess_point_cloud(pcd, voxel_size):
    # Downsample the point cloud
    pcd_down = pcd.voxel_down_sample(voxel_size)

    # Estimate normals
    pcd_down.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30))

    # Compute FPFH feature
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100)
    )

    return pcd_down, fpfh

def estimate_initial_guess_fpfh(source, target, voxel_size=0.05):
    source_down, source_fpfh = preprocess_point_cloud(source, voxel_size)
    target_down, target_fpfh = preprocess_point_cloud(target, voxel_size)

    # RANSAC-based global registration
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down, target_down, source_fpfh, target_fpfh,
        mutual_filter=True, max_correspondence_distance=voxel_size * 1.5,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        ransac_n=4, criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500)
    )

    return result.transformation


In [5]:
import copy
def draw_registration_result(source, target, transformation):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    # source_temp.paint_uniform_color([1, 0.706, 0])
    # target_temp.paint_uniform_color([0, 0.651, 0.929])
    source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp],
                                      zoom=0.4459,
                                      front=[0.9288, -0.2951, -0.2242],
                                      lookat=[1.6784, 2.0612, 1.4451],
                                      up=[-0.3402, -0.9189, -0.1996])

In [37]:
point_cloud_files = os.listdir("output_pcds")
point_cloud_files = [f"output_pcds/{f}" for f in point_cloud_files]
demo_icp_pcds = [point_cloud_files[0], point_cloud_files[1]]

pcd1 = o3d.io.read_point_cloud(point_cloud_files[0])
pcd2 = o3d.io.read_point_cloud(point_cloud_files[1])

threshold = 0.6

# Generate a valid rotation matrix
# R = special_ortho_group.rvs(3)  # 3x3 orthonormal rotation
# t = np.array([0.5, 0.2, 0.3])  # Approximate initial translation

# initial_guess = np.eye(4)
# initial_guess[:3, :3] = R
# initial_guess[:3, 3] = t


# initial_guess = np.eye(4)

# Compute FPFH + RANSAC initial transformation
# initial_guess = estimate_initial_guess_fpfh(pcd1, pcd2)
initial_guess = [[-1.77804560e-01, -9.84064725e-01,  1.46799224e-03, -7.99785952e-02],
 [ 9.84065463e-01, -1.77805569e-01, -5.86609758e-04,  5.89522761e-02],
 [ 8.38279166e-04,  1.34029858e-03,  9.99998750e-01, -7.47250756e-03],
 [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  1.00000000e+00]]

print("Estimated Initial Guess (Using FPFH + RANSAC):\n", initial_guess)

# draw_registration_result(pcd1, pcd2, initial_guess)

Estimated Initial Guess (Using FPFH + RANSAC):
 [[-0.17780456, -0.984064725, 0.00146799224, -0.0799785952], [0.984065463, -0.177805569, -0.000586609758, 0.0589522761], [0.000838279166, 0.00134029858, 0.99999875, -0.00747250756], [0.0, 0.0, 0.0, 1.0]]


In [38]:
print("Initial alignment")
evaluation = o3d.pipelines.registration.evaluate_registration(
    pcd1, pcd2, threshold, initial_guess)
print(evaluation)

Initial alignment
RegistrationResult with fitness=6.952348e-01, inlier_rmse=2.574646e-01, and correspondence_set size of 15932
Access transformation to get result.


In [40]:
trans_init = initial_guess
print("Apply point-to-point ICP")
reg_p2p = o3d.pipelines.registration.registration_icp(
    pcd1, pcd2, threshold, trans_init,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000))
print(reg_p2p)
print("Transformation is:")
print(reg_p2p.transformation)
# draw_registration_result(pcd1, pcd2, reg_p2p.transformation)

Apply point-to-point ICP
RegistrationResult with fitness=7.340723e-01, inlier_rmse=2.574380e-01, and correspondence_set size of 16822
Access transformation to get result.
Transformation is:
[[-0.15223289 -0.9868462   0.05440335  0.12847902]
 [ 0.98748635 -0.1541636  -0.03323076  0.05340107]
 [ 0.04118067  0.04866375  0.99796593 -0.11269197]
 [ 0.          0.          0.          1.        ]]


In [56]:
trans_init = reg_p2p.transformation
print("Apply point-to-point ICP")
reg_p2p = o3d.pipelines.registration.registration_icp(
    pcd1, pcd2, threshold, trans_init,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000))
print(reg_p2p)
print("Transformation is:")
print(reg_p2p.transformation)

# draw_registration_result(pcd1, pcd2, reg_p2p.transformation)

Apply point-to-point ICP
RegistrationResult with fitness=9.098447e-02, inlier_rmse=1.099511e-02, and correspondence_set size of 2085
Access transformation to get result.
Transformation is:
[[ 0.9999969   0.00214562  0.0012634   0.00561848]
 [-0.00214831  0.99999541  0.00213575  0.02289774]
 [-0.00125882 -0.00213846  0.99999692 -0.0016348 ]
 [ 0.          0.          0.          1.        ]]


In [21]:




trans_init = np.asarray([[0.862, 0.011, -0.507, 0.5],
                         [-0.139, 0.967, -0.215, 0.7],
                         [0.487, 0.255, 0.835, -1.4], [0.0, 0.0, 0.0, 1.0]])

point_cloud_files = os.listdir("output_pcds")
point_cloud_files = [f"output_pcds/{f}" for f in point_cloud_files]
demo_icp_pcds = [point_cloud_files[0], point_cloud_files[1]]

pcd1 = o3d.io.read_point_cloud(point_cloud_files[0])
pcd2 = o3d.io.read_point_cloud(point_cloud_files[1])

# Downsample for faster ICP
pcd1 = pcd1.voxel_down_sample(voxel_size=0.02)
pcd2 = pcd2.voxel_down_sample(voxel_size=0.02)


# initial_guess = np.eye(4) 

# Perform Point-to-Point ICP
reg_p2p = o3d.pipelines.registration.registration_icp(
    pcd2, pcd1, threshold, trans_init,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000)
)

# Print the estimated transformation matrix
print("Estimated Transformation Matrix:")
print(reg_p2p.transformation)

# Visualize the registration result
# pcd2.transform(reg_p2p.transformation)  # Apply transformation to align pcd2 to pcd1
# o3d.visualization.draw_geometries([pcd1, pcd2], window_name="ICP Registration")
draw_registration_result(pcd2, pcd1, reg_p2p.transformation)


Estimated Transformation Matrix:
[[ 0.86204971  0.01290489 -0.50686902  0.49344043]
 [-0.14313435  0.9656128  -0.21851036  0.70753005]
 [ 0.48571273  0.2601161   0.83416786 -1.41587664]
 [ 0.          0.          0.          1.        ]]


error: XDG_RUNTIME_DIR not set in the environment.
